# 01b · Evaluating principal-risk extraction

Scores one saved extraction run (`runs/extract/`, made by notebook 01) against the hand-labelled golden set
(`eval/golden/risk_extraction.yaml`), page by page.

- **How:** for each page, an LLM judge gets the golden and the predicted risks, **pairs** the ones that describe the
  same risk, and grades each pair's title, section, description and mitigation (`extract-judge`). Category is graded in
  code: it is an exact label, so it needs no judge.
- **What we report:** recall, precision and F1, overall and per page. **Recall leads** (plan: a missed principal
  risk costs more than an extra one the analyst rejects). The field grades are saved with the results, for later.
- **No-risk pages count:** p.50 and p.72 are labelled with no principal risks, so anything extracted there is a
  false positive.

The code lives in `annual_report_reader/evaluation/`, one module per section (named in its heading), and is
unit-tested (`tests/evaluation/`). This notebook runs each stage on the latest extraction run.

## 1. Config → `evaluation/config.py`

The judge should not be the model it grades: it is the Sol deployment, configurable, so a judge from another
provider can be dropped in later. Sol's price is not known yet, so cost is shown only once the prices are set.
The run to score is named explicitly (`EXTRACTION_RUN`): there is no "latest", because the newest run is often an
experiment (another model, another prompt). `JudgeConfig` holds only what decides a grade, so the same config
grades any run.

In [1]:
from dotenv import load_dotenv

from annual_report_reader.evaluation.config import JudgeConfig
from annual_report_reader.extraction.config import REPO_ROOT

load_dotenv(REPO_ROOT / ".env")  # endpoint, key and deployment names

EXTRACTION_RUN = "extract-20261112T000123Z"  # Luna, high effort, prompt extract
JUDGE_CONFIG = JudgeConfig()
JUDGE_CONFIG

JudgeConfig(golden_path=PosixPath('eval/golden/risk_extraction.yaml'), judge_deployment='gpt-5.6-sol-carl-test', judge_effort='medium', judge_prompt_dir=PosixPath('prompts'), judge_prompt='extract-judge', max_concurrency=4, price_in_per_m=None, price_out_per_m=None)

## 2. Golden set → `evaluation/golden.py`

One entry per risk **per page**, as step 1 should return it: a risk mentioned on two pages is expected on both.
A page listed as `[]` is labelled and has no principal risks. The category labels are the extraction's own
`Category`, and unknown keys are rejected, so a typo in the golden file fails here instead of silently never matching.

In [2]:
from annual_report_reader.evaluation.golden import load_golden

golden = load_golden(JUDGE_CONFIG.resolve(JUDGE_CONFIG.golden_path))
print(f"{sum(len(rs) for rs in golden.values())} golden risks on {len(golden)} labelled pages")
for page, risks in golden.items():
    print(f"  p.{page:<4} {len(risks)}  " + " | ".join(r.title for r in risks))

14 golden risks on 8 labelled pages
  p.50   0  
  p.51   4  Geopolitical conflict and trade barriers | Shifting industrial policy and regulation | Project execution | Cyber attacks
  p.71   1  Carbon taxes and tariffs
  p.72   0  
  p.73   3  Cost implication of injuries in own workforce | Cost implications of injuries to contractors and sub-contractors | Fines related to forced or child labour
  p.74   3  Insufficient market conditions | Cyber security risks | Risk of corruption and bribery
  p.85   1  Carbon taxes and tariffs
  p.118  2  Cyber security incidents | Cyber security risks


## 3. The extraction run → `evaluation/coverage.py`

Only pages that are both **labelled** and **sent** can be scored. The others are listed, not scored:

- **not labelled:** extracted, but the golden set says nothing about the page;
- **not sent:** labelled, but the run did not extract it (a config choice, not a pipeline error).

A page whose extraction call **failed** is scored: its golden risks count as missed, because that is what the
analyst would get.

In [3]:
from annual_report_reader.evaluation.coverage import coverage
from annual_report_reader.extraction.runs import RUNS_DIR, load_run

run = load_run(REPO_ROOT / RUNS_DIR / f"{EXTRACTION_RUN}.json")
cov = coverage(run, golden)
SCORED_PAGES = cov.index[cov.scored].tolist()
print(f"{run.run_id}: {len(run.risks)} risks, prompt {run.config.prompt}, model {run.config.deployment}")
cov.T

extract-20261112T000123Z: 12 risks, prompt extract, model gpt-5.6-luna-carl-test


page,50,51,71,72,73,74,85,118
golden,0,4,1,0,3,3,1,2
status,ok,ok,ok,ok,ok,ok,ok,ok
scored,True,True,True,True,True,True,True,True


## 4. Judge → `evaluation/judge.py`

One call per page. The judge gets both lists as JSON, each risk with an id and the four fields it grades. Category
and confidence are left out, so they cannot bias the matching. The prompt is a file
(`prompts/extract-judge.md`), sent unchanged; the page's JSON is the user message.

The answer is strict structured output: a list of pairs (`golden_id` ↔ `pred_id`), each with four `yes`/`no`
grades and a one-sentence reason.

In [4]:
# Judge one page (p.51): the raw verdicts, before any checks.
from annual_report_reader.chat import Usage, azure_chat_model
from annual_report_reader.evaluation.judge import JUDGE_FIELDS, judge_chain, load_judge_prompt, page_cases

cases = page_cases(run, golden, SCORED_PAGES)
judge_prompt = load_judge_prompt(JUDGE_CONFIG.resolve(JUDGE_CONFIG.judge_prompt_dir), JUDGE_CONFIG.judge_prompt)
judge = judge_chain(judge_prompt, azure_chat_model(JUDGE_CONFIG.judge_deployment, JUDGE_CONFIG.judge_effort))

case = next(c for c in cases if c.page == 51)
out = judge.invoke(case)
print(f"p.{case.page}: {len(case.golden)} golden, {len(case.predicted)} predicted | {Usage.of(out['raw'])}")
for v in out["parsed"].pairs:
    grades = " ".join(f"{f}={getattr(v, f)}" for f in JUDGE_FIELDS)
    print(f"  g{v.golden_id} {case.golden[v.golden_id].title!r} ↔ p{v.pred_id} {case.predicted[v.pred_id].title!r}")
    print(f"     {grades} | {v.reason}")

p.51: 4 golden, 3 predicted | input_tokens=2444 output_tokens=341 reasoning_tokens=137
  g0 'Geopolitical conflict and trade barriers' ↔ p0 'Geopolitics and regulatory framework'
     title=no section=yes description=no mitigation=no | The prediction includes the geopolitical conflict and trade-barrier risk and points to the correct main-risk entry, but its title is broader and its description and mitigation also merge in the separate industrial-policy risk and advocacy action.
  g2 'Project execution' ↔ p1 'Project execution'
     title=yes section=yes description=yes mitigation=yes | The prediction accurately reproduces the project execution risk, its location, all causes and consequences, and both mitigation actions.
  g3 'Cyber attacks' ↔ p2 'Cyber attacks'
     title=yes section=yes description=yes mitigation=yes | The prediction matches the cyber-attack risk, section, full description, and mitigation, with only inconsequential spelling and formatting differences.


## 5. Pairs → `evaluation/judge.py`

The judge's pairs must be one-to-one with valid ids, which a JSON schema cannot enforce, so code checks it: the first
valid pair wins, and any other pair is **dropped with its reason** and kept in the results, so judge sloppiness
stays visible. Each kept pair also gets the code's category grades:

- `category`: the predicted primary category is the golden primary;
- `category_any`: the predicted primary is the golden primary or secondary (close calls such as regulatory vs
  market are common).

A page where one side is empty needs no judge: every golden risk is missed, or every prediction is extra.
A page whose judge call fails is **not scored** and is flagged (an evaluation failure, not a pipeline one).

In [5]:
# The same verdicts after the checks: kept pairs with the code's category grades, and anything dropped.
from annual_report_reader.evaluation.judge import page_result

checked = page_result(case, out)
for p in checked.pairs:
    print(f"  g{p.golden_id} ↔ p{p.pred_id} | category={p.category} category_any={p.category_any}")
for d in checked.dropped:
    print(f"  dropped g{d.verdict.golden_id} ↔ p{d.verdict.pred_id}: {d.why}")

  g0 ↔ p0 | category=no category_any=no
  g2 ↔ p1 | category=yes category_any=yes
  g3 ↔ p2 | category=yes category_any=yes


## 6. Judge every page → `evaluation/judge.py`

Pages with both golden and predicted risks are judged concurrently (`batch`); the others are decided in code.

In [6]:
import time
from datetime import UTC, datetime

from annual_report_reader.evaluation.judge import eval_id_for, judge_pages

started_at = datetime.now(UTC)
eval_id = eval_id_for(run.run_id, JUDGE_CONFIG.judge_prompt, started_at)
start = time.perf_counter()
page_results = judge_pages(judge, cases, JUDGE_CONFIG, eval_id)
elapsed = time.perf_counter() - start

for r in page_results:
    how = f"JUDGE FAILED: {r.judge_error}" if r.judge_error else "judged" if r.judged else "decided in code"
    extra = f" | {len(r.dropped)} dropped" if r.dropped else ""
    print(f"p.{r.page:<4} | {len(r.golden)} golden, {len(r.predicted)} predicted, {len(r.pairs)} paired{extra} | {how}")
judge_usage = sum((r.usage for r in page_results), Usage())
cost = JUDGE_CONFIG.cost(judge_usage)
print(f"{eval_id}: {elapsed:.0f}s | {judge_usage} | cost: {'unknown' if cost is None else f'${cost:.4f}'}")

p.50   | 0 golden, 0 predicted, 0 paired | decided in code
p.51   | 4 golden, 3 predicted, 3 paired | judged
p.71   | 1 golden, 1 predicted, 1 paired | judged
p.72   | 0 golden, 0 predicted, 0 paired | decided in code
p.73   | 3 golden, 3 predicted, 3 paired | judged
p.74   | 3 golden, 3 predicted, 3 paired | judged
p.85   | 1 golden, 1 predicted, 1 paired | judged
p.118  | 2 golden, 1 predicted, 1 paired | judged
extract-20261112T000123Z--judge-v1--20261112T003630Z-6521: 4s | input_tokens=7666 output_tokens=1147 reasoning_tokens=283 | cost: unknown


## 7. Scores → `evaluation/scores.py`

- **recall** = paired / golden: the share of principal risks found;
- **precision** = paired / predicted: the share of predictions that are principal risks;
- **F1** combines both.

Overall scores are **micro-averaged** (counts summed over pages), so a page with four risks weighs four times a page
with one. A page with nothing on one side has no recall or precision (shown as empty); p.50 and p.72 still count
towards overall precision, which is the point of labelling them. Pages whose judge failed are left out.

In [7]:
from annual_report_reader.evaluation.scores import scores_table, total

if failed := [r.page for r in page_results if r.judge_error]:
    print(f"PARTIAL SCORES: judge failed on pages {failed}; they are left out.")
t = total(page_results)
print(
    f"recall {t.recall:.2f} | precision {t.precision:.2f} | F1 {t.f1:.2f}  ({t.paired} of {t.golden} golden found, "
    f"{t.predicted - t.paired} extra)"
)
table = scores_table(page_results).round(2)
table.astype(object).where(table.notna(), "")  # no recall / precision on a page: shown empty

recall 0.86 | precision 1.00 | F1 0.92  (12 of 14 golden found, 0 extra)


,golden,predicted,paired,recall,precision,f1
p.50,0,0,0,,,
p.51,4,3,3,0.75,1.0,0.86
p.71,1,1,1,1.0,1.0,1.0
p.72,0,0,0,,,
p.73,3,3,3,1.0,1.0,1.0
p.74,3,3,3,1.0,1.0,1.0
p.85,1,1,1,1.0,1.0,1.0
p.118,2,1,1,0.5,1.0,0.67
all,14,12,12,0.86,1.0,0.92


## 8. Save → `evaluation/runs.py`

Saved to `eval/runs/<extraction run>--<judge prompt>--<time>.json`: the config, the judge prompt as sent, the
coverage, every page with its pairs, dropped pairs and usage, and the overall counts. Every evaluation is kept.

In [8]:
from annual_report_reader.evaluation.runs import EVAL_RUNS_DIR, EvaluationRun, load_eval, save_eval

evaluation = EvaluationRun(
    eval_id=eval_id,
    started_at=started_at,
    elapsed_s=round(elapsed, 1),
    config=JUDGE_CONFIG,
    extraction_run_id=run.run_id,
    judge_system_prompt=judge_prompt.format_messages(payload="")[0].content,
    coverage=cov.status.to_dict(),
    pages=page_results,
)
eval_path = save_eval(evaluation, REPO_ROOT / EVAL_RUNS_DIR)
assert load_eval(eval_path).total == evaluation.total  # the file loads back
print(f"Saved {eval_path.relative_to(REPO_ROOT)}")

Saved eval/runs/extract-20261112T000123Z--judge-v1--20261112T003630Z-6521.json


## 9. Observations

_To fill in after reviewing the scores._